# 01 - Orbit Usage

This notebook is the first skill in the Nebula learning path.

You will learn how to:
- create two-body and numerical orbits
- query state in multiple frames
- use multiple time input types
- use convenience state accessors


In [ ]:
import numpy as np
import astropy.units as u
from astropy.time import Time

from nebula.propagation import Orbit, initialize_orekit
from nebula.time_utils import astropy_time_to_orekit_date

np.set_printoptions(precision=6, suppress=True)
initialize_orekit()


In [ ]:
# 1) Build a two-body analytical orbit
epoch = Time("2026-01-01T00:00:00", scale="utc")
orbit = Orbit.from_kepler_two_body(
    epoch=epoch,
    a=7000e3,
    e=0.001,
    i=np.deg2rad(53.0),
    raan=np.deg2rad(20.0),
    argp=np.deg2rad(15.0),
    anomaly=np.deg2rad(10.0),
    anomaly_type="mean",
)

orbit.epoch


In [ ]:
# 2) Query position/velocity/acceleration at absolute times
times = Time(epoch.unix + np.arange(0, 301, 60, dtype=np.float64), format="unix", scale="utc")
p_gcrf, v_gcrf, a_gcrf = orbit.get_pva(times, frame="gcrf")
p_itrf = orbit.get_p(times, frame="itrf")
lat, lon, alt = orbit.get_geodetic(times)

print("p_gcrf shape:", p_gcrf.shape)
print("p_itrf shape:", p_itrf.shape)
print("first geodetic sample [deg, deg, m]:", lat[0].value, lon[0].value, alt[0].value)


In [ ]:
# 3) Time input options: astropy Time, seconds from epoch, Quantity, AbsoluteDate
t_astropy = Time(epoch.unix + np.array([0.0, 30.0, 60.0]), format="unix", scale="utc")
t_seconds = np.array([0.0, 30.0, 60.0], dtype=np.float64)
t_quantity = t_seconds * u.s
t0 = astropy_time_to_orekit_date(epoch)
t_absolute = [t0.shiftedBy(0.0), t0.shiftedBy(30.0), t0.shiftedBy(60.0)]

r_astropy = orbit.get_p_np(t_astropy, frame="gcrf")
r_seconds = orbit.get_p_np(t_seconds, frame="gcrf")
r_quantity = orbit.get_p_np(t_quantity, frame="gcrf")
r_absolute = orbit.get_p_np(t_absolute, frame="gcrf")

print("allclose(astropy, seconds):", np.allclose(r_astropy, r_seconds))
print("allclose(seconds, quantity):", np.allclose(r_seconds, r_quantity))
print("allclose(seconds, AbsoluteDate):", np.allclose(r_seconds, r_absolute))


In [ ]:
# 4) Build a higher-fidelity numerical orbit with common perturbations
orbit_num = Orbit.from_kepler_numerical(
    epoch=epoch,
    a=7050e3,
    e=0.002,
    i=np.deg2rad(97.4),
    raan=np.deg2rad(5.0),
    argp=np.deg2rad(45.0),
    anomaly=np.deg2rad(0.0),
    anomaly_type="mean",
    gravity_degree=12,
    gravity_order=12,
    enable_third_body=True,
    third_bodies=("sun", "moon"),
    enable_srp=True,
    srp_area_m2=1.0,
    srp_cr=1.2,
    enable_drag=False,
)

state = orbit_num.get_state(np.array([0.0, 120.0, 240.0]), frame="gcrf", fields="pv", as_quantity=False)
print("numerical p shape:", state["p"].shape, "v shape:", state["v"].shape)


Next notebook: **02 - Transforms Usage**.
